In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pickle

In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")

df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

X = df.drop(columns=['loan_status'])
y = df['loan_status']

categorical_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())                   
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder', OneHotEncoder(handle_unknown='ignore'))   
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=5, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Точність моделі (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

with open('credit_risk_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Модель успішно збережено у файл 'credit_risk_model.pkl'")

Точність моделі (Accuracy): 0.9372
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      4971
           1       0.97      0.73      0.83      1365

    accuracy                           0.94      6336
   macro avg       0.95      0.86      0.90      6336
weighted avg       0.94      0.94      0.93      6336

Модель успішно збережено у файл 'credit_risk_model.pkl'
